# Figure 5 — analysis (PSC → cardiomyocyte)
H9 hESC day 0 vs ventricular cardiomyocytes day 80 (GSE116862, Zhang et al. 2019), 2 biological replicates per state.
Computes every Figure 5 / Supplementary Figure 5Sa / 5Sb result and writes it to `data/`; the panels are drawn from these
tables in `psccm_figures.ipynb` and assembled in `psccm_pages.ipynb`. Reads the GSEA tables written by
`psccm_expression.ipynb` (run that first) and the LAS graphs under `/data/land/3d-inter-paper-data/`. Genes sit on their
TSS bin (`psccm_genes.py`, hg19). Runs in ~4 min.

*Note on re-reads.* Some tables are read back from `data/` on purpose, so every published number stays bit-identical to
the published tables, which were built from re-read CSVs. pandas' default CSV parser does not always return the exact
float that was written: for these tables it differs from the in-memory value by up to ~2e-16 relative for folds / NES
and up to ~9e-13 relative for small p / q values, far below any reported precision. Hence:
- the setup cell reads the GSEA tables with `float_precision="round_trip"`, which returns exactly the values
  `psccm_expression.ipynb` computed (the published tables joined them in memory);
- `colocalization.csv` is re-read with the default parser right after it is written (the published notebooks read it
  back);
- the supplementary-tables section reads `gsea_celltype.csv` and `supp_coloc_vs_expression_table.csv` with the default
  parser, as the figures notebook does (the published tables were built that way).

Do not "harmonize" these parsers: that would change the last digits of the published tables.

In [1]:
import json
import os
import pickle
import sys

import networkx as nx
import pandas as pd
from scipy.stats import false_discovery_control, spearmanr

sys.path.insert(0, "..")                                # shared modules
import coloc_stats                                      # contact counting, per-state hypergeometric test + BH, Fisher test
import kcore                                            # state-specific k-cores
from psccm_genes import use_tss_genes                   # one TSS bin per protein-coding gene (hg19)

DATA = "data"
META = "../../metadata"
RUNS = "/data/land/3d-inter-paper-data/psc-cm-gse116862/runs"
TSS_BINS = f"{META}/hg19/bins_df_protein_coding_genes_with_tss_hg19_250000.csv"   # chrom per graph node
GSEA_COLUMNS = ["Term", "NES", "NOM p-val", "FDR q-val"]

SETS = {"hallmark": json.load(open(f"{META}/hallmark_and_custom_pathways_unfiltered_aug_2025.json")),
        "cardiomyocyte": json.load(open(f"{META}/psc-cm/cardiomyocyte_published_sets.json")),
        "stemness": json.load(open(f"{META}/psc-cm/stemness_published_sets.json")),
        "celltype": json.load(open(f"{META}/psc-cm/panglaodb_celltypes.json"))}
panel = json.load(open(f"{META}/psc-cm/psc_cm_pathways.json"))["cardiomyocyte"]      # main panel (5b): name -> genes
panel_names = list(panel)
# GSEA tables from psccm_expression.ipynb, round_trip = exactly as computed (see the note on re-reads)
gsea_results = {name: pd.read_csv(f"{DATA}/gsea_{name}.csv", float_precision="round_trip")
                for name in ["hallmark", "cardiac", "stemness", "celltype"]}
print({c: len(s) for c, s in SETS.items()})

{'hallmark': 60, 'cardiomyocyte': 14, 'stemness': 13, 'celltype': 178}


## Colocalization test (Figure 5b–d; Supplementary Figure 5Sa, panels b–d; 5Sb, panel a)
Per-state hypergeometric test (shared `../coloc_stats.py`) of every gene set on the pooled day00-vs-day80 LAS graph and
the 6 replicate-pair graphs (4 cross, 2 within), with each state's fold enrichment and the Fisher differential test.
Writes `data/colocalization.csv`.

In [2]:
# comparison -> (state A, state B, kind, graph)
COMPARISONS = {"pooled": ("day00", "day80", "pooled", f"{RUNS}/pooled-v2/analysis/G_all_day00_vs_day80_filt_20.0.pkl")}
for a, b in [("day00_R1", "day80_R1"), ("day00_R1", "day80_R2"), ("day00_R2", "day80_R1"), ("day00_R2", "day80_R2"),
             ("day00_R1", "day00_R2"), ("day80_R1", "day80_R2")]:
    kind = "within" if a[:5] == b[:5] else "cross"       # same state (dayNN) -> within
    COMPARISONS[f"{a} vs {b}"] = (a, b, kind, f"{RUNS}/replicate-resolved/analysis/G_all_{a}_vs_{b}_filt_20.0.pkl")

graphs = {comp: use_tss_genes(pickle.load(open(path, "rb"))) for comp, (_, _, _, path) in COMPARISONS.items()}

In [3]:
rows = []
for comp, (a, b, kind, _) in COMPARISONS.items():
    G = graphs[comp]
    bg = coloc_stats.count_contacts(G, "genes")          # genome-wide background
    M = bg[a] + bg[b] + bg["Shared"]
    for category, sets in SETS.items():
        for name, genes in sets.items():
            c = coloc_stats.count_contacts(G, "genes", genes)
            ka = c.get(a, 0)
            kb = c.get(b, 0)
            N = ka + kb + c.get("Shared", 0)
            if N == 0:
                continue
            fold_a = (ka / N) / (bg[a] / M)
            fold_b = (kb / N) / (bg[b] / M)
            p_a = coloc_stats.hypergeom_enrichment_pval(ka, M, bg[a], N)
            p_b = coloc_stats.hypergeom_enrichment_pval(kb, M, bg[b], N)
            fisher_or, fisher_p = coloc_stats.differential_fisher(kb, ka, bg[b], bg[a])   # odds ratio = fold_b / fold_a
            rows.append(dict(comparison=comp, kind=kind, state_a=a, state_b=b, gene_set=name, category=category,
                             N=N, k_a=ka, k_b=kb, fold_a=fold_a, fold_b=fold_b, p_a=p_a, p_b=p_b,
                             fisher_or=fisher_or, fisher_p=fisher_p))
    print(f"{comp}: background {M:,} gene pairs")

pd.DataFrame(rows).to_csv(f"{DATA}/colocalization.csv", index=False)
coloc = pd.read_csv(f"{DATA}/colocalization.csv")       # re-read, default parser (see the note on re-reads)
print(f"wrote data/colocalization.csv: {len(coloc)} rows")

pooled: background 23,450,263 gene pairs


day00_R1 vs day80_R1: background 25,062,565 gene pairs


day00_R1 vs day80_R2: background 25,012,336 gene pairs


day00_R2 vs day80_R1: background 25,259,717 gene pairs


day00_R2 vs day80_R2: background 25,199,837 gene pairs


day00_R1 vs day00_R2: background 20,498,807 gene pairs


day80_R1 vs day80_R2: background 22,121,691 gene pairs
wrote data/colocalization.csv: 1855 rows


## Joint FDR and gene-set tables (Figure 5b, 5c; Supplementary Figure 5Sa, panels c, d; 5Sb)
Per-state BH over every pathway-level set tested (60-pathway panel + 14 cardiac + 13 stemness = 87 sets), so a set has
one q in every panel; joined with GSEA. Writes `data/panel_table.csv` (5b), `data/supp_cardiac_sets_table.csv` and
`data/supp_stemness_sets_table.csv` (5Sa c, d), and `data/supp_coloc_vs_expression_table.csv` (5c, 5Sb).

In [4]:
pooled = coloc[coloc.comparison == "pooled"].set_index("gene_set")
cross = coloc[coloc.kind == "cross"]


def pooled_table(category):
    """Pooled folds / p-values (day00 = A, day80 = B) + number of cross-replicate pairs with p < 0.05."""
    t = pooled[pooled.category == category]
    reps = cross[cross.category == category].groupby("gene_set")[["p_a", "p_b"]].agg(lambda p: (p < 0.05).sum())
    table = pd.DataFrame({"N": t.N, "fold_day00": t.fold_a, "fold_day80": t.fold_b, "p_day00": t.p_a, "p_day80": t.p_b,
                          "rep_sig_day00_of4": reps.p_a, "rep_sig_day80_of4": reps.p_b})
    return table.rename_axis("gene_set").reset_index()


hall = pooled_table("hallmark")
cardiac = pooled_table("cardiomyocyte")
stem = pooled_table("stemness")
joint = pd.concat([hall, cardiac, stem], ignore_index=True)
for s in ["day00", "day80"]:
    joint[f"q_{s}"] = coloc_stats.bh(joint[f"p_{s}"])
q = joint.set_index("gene_set")[["q_day00", "q_day80"]]
cardiac = cardiac.join(q, on="gene_set")
stem = stem.join(q, on="gene_set")
print(f"joint FDR family: {len(joint)} sets")

joint FDR family: 87 sets


In [5]:
def with_gsea(table, results):
    """Colocalization table + GSEA NES / p / FDR per set, ordered by CM fold."""
    g = results[GSEA_COLUMNS].rename(columns={"Term": "gene_set"})
    return table.merge(g, on="gene_set").sort_values("fold_day80").reset_index(drop=True)


# 5b: main panel (5 of the published cardiac sets; GSEA from the cardiac-set run, so a set has one q everywhere)
panel_table = with_gsea(cardiac[cardiac.gene_set.isin(panel_names)], gsea_results["cardiac"])
panel_table = panel_table.rename(columns={"NES": "nes", "NOM p-val": "nes_pval", "FDR q-val": "nes_fdr"})
panel_table.to_csv(f"{DATA}/panel_table.csv", index=False)

# 5Sa c, d: all published cardiac / stemness sets
with_gsea(cardiac, gsea_results["cardiac"]).to_csv(f"{DATA}/supp_cardiac_sets_table.csv", index=False)
with_gsea(stem, gsea_results["stemness"]).to_csv(f"{DATA}/supp_stemness_sets_table.csv", index=False)

# 5c, 5Sb: 60-pathway panel + main-panel sets, colocalization (joint q) + expression
cardiac_panel_gsea = gsea_results["cardiac"][gsea_results["cardiac"].Term.isin(panel_names)]
expr = pd.concat([gsea_results["hallmark"], cardiac_panel_gsea])[["Term", "NES", "FDR q-val"]]
expr = expr.drop_duplicates("Term")                     # defensive: no panel set shares a name with the 60 pathways
expr = expr.rename(columns={"Term": "name", "NES": "expr_nes", "FDR q-val": "expr_fdr"})
pathways = joint[joint.gene_set.isin(set(hall.gene_set) | set(panel_names))]
pathways = pathways.rename(columns={"gene_set": "name", "q_day00": "fdr_day00", "q_day80": "fdr_day80"})
pathways = pathways[["name", "fold_day00", "fold_day80", "p_day00", "p_day80", "fdr_day00", "fdr_day80"]]
pathways = pathways.merge(expr, on="name")
pathways.to_csv(f"{DATA}/supp_coloc_vs_expression_table.csv", index=False)
panel_table[["gene_set", "N", "fold_day00", "fold_day80", "q_day80", "rep_sig_day80_of4", "nes", "nes_fdr"]]

,gene_set,N,fold_day00,fold_day80,q_day80,rep_sig_day80_of4,nes,nes_fdr
0,Adrenergic signaling in cardiomyocytes (KEGG),1835,0.973081,1.126225,3.438617e-04,3,2.140885,0.0
1,Muscle Contraction (Reactome),2383,1.010658,1.205638,2.772437e-11,4,2.307842,0.0
2,Hypertrophic cardiomyopathy (KEGG),720,1.056603,1.217966,1.258378e-04,4,2.155260,0.0
3,Cardiomyocytes (PanglaoDB),2031,0.648222,1.411572,7.431703e-35,4,3.042274,0.0
4,Sarcomere organization (GO:0045214),57,0.622379,1.990345,2.874145e-06,4,2.385352,0.0


## Supplementary tables: per-state enrichment and Fisher test (reference tables for Figure 5c, 5d; not plotted)
Reference tables, not plotted: per-state enrichment FDRs and the Fisher differential test (CM- vs PSC-specific contacts
vs the genome, BH within each table) for the 178 PanglaoDB cell types (5d) and the 65 pathways of 5c. Writes
`data/supp_table_celltypes_perstate_and_fisher.csv` and `data/supp_table_pathways_perstate_and_fisher.csv`.

In [6]:
celltypes = coloc.query("category == 'celltype' and comparison == 'pooled'")
# default parser on purpose, as the figures read it, not gsea_results["celltype"] (see the note on re-reads)
celltype_gsea = pd.read_csv(f"{DATA}/gsea_celltype.csv").rename(columns={"Term": "gene_set"})
celltypes = celltypes.merge(celltype_gsea[["gene_set", "NES"]], on="gene_set")
celltype_table = celltypes.assign(CM_enrichment_fdr=false_discovery_control(celltypes.p_b, method="bh"),
                                  PSC_enrichment_fdr=false_discovery_control(celltypes.p_a, method="bh"),
                                  fisher_fdr=false_discovery_control(celltypes.fisher_p, method="bh"))
celltype_table = celltype_table[["gene_set", "N", "k_b", "k_a", "fold_b", "fold_a", "CM_enrichment_fdr",
                                 "PSC_enrichment_fdr", "fisher_or", "fisher_p", "fisher_fdr", "NES"]]
celltype_table = celltype_table.rename(columns={"k_b": "n_CM_only", "k_a": "n_PSC_only", "fold_b": "CM_fold",
                                                "fold_a": "PSC_fold", "fisher_or": "fisher_odds_ratio", "NES": "GSEA_NES"})
celltype_table.sort_values("fisher_p").to_csv(f"{DATA}/supp_table_celltypes_perstate_and_fisher.csv", index=False)
print(f"{len(celltype_table)} cell types")

178 cell types


In [7]:
# the 65 pathways of 5c (60-pathway panel + 5 main-panel sets) on the pooled graph
# re-read as saved, default parser, as the figures read it (see the note on re-reads)
pathways = pd.read_csv(f"{DATA}/supp_coloc_vs_expression_table.csv")
pathway_coloc = coloc.query("category == 'hallmark' or gene_set in @panel_names")      # every comparison
pw = pathway_coloc[pathway_coloc.comparison == "pooled"].merge(pathways[["name", "expr_nes", "expr_fdr"]],
                                                         left_on="gene_set", right_on="name")
pw = pw.merge(pathways[["name", "fdr_day00", "fdr_day80"]], on="name")
pw["fisher_fdr"] = false_discovery_control(pw.fisher_p, method="bh")
pathway_table = pw[["gene_set", "N", "k_b", "k_a", "fold_b", "fold_a", "fdr_day80", "fdr_day00", "fisher_or", "fisher_p",
                    "fisher_fdr", "expr_nes", "expr_fdr"]]
pathway_table = pathway_table.rename(columns={"k_b": "n_CM_only", "k_a": "n_PSC_only", "fold_b": "CM_fold",
                                              "fold_a": "PSC_fold", "fdr_day80": "CM_enrichment_fdr",
                                              "fdr_day00": "PSC_enrichment_fdr", "fisher_or": "fisher_odds_ratio",
                                              "expr_nes": "GSEA_NES", "expr_fdr": "GSEA_fdr"})
pathway_table.sort_values("fisher_p").to_csv(f"{DATA}/supp_table_pathways_perstate_and_fisher.csv", index=False)
print(f"{len(pathway_table)} pathways")

65 pathways


## State-specific k-cores (Figure 5e, 5f)
Densest k-core of the CM-only (day 80) colocalization graph among a gene set's genes on the pooled graph (shared
`../kcore.py`), with the gene × gene contact-state matrix, genes in genomic order. Writes
`data/kcores/{sarcomere,cardiomyocyte}_{genes,states}.csv` and `data/kcores/kcore_summary.csv`.

In [8]:
# pooled LAS graph: edge type per bin pair; gene -> its TSS bin
G = graphs["pooled"]
edge_type = nx.get_edge_attributes(G, "edge_type")
gene_bins = {}
for b, genes in nx.get_node_attributes(G, "genes").items():
    for g in genes:
        gene_bins.setdefault(g, set()).add(b)
bin_chrom = {b: int(c) for b, c in pd.read_csv(TSS_BINS, index_col=0)["chrom"].items()}   # row index = graph node id
STATE_CODE = {"day80": 1, "day00": 2, "Shared": 3}    # heatmap codes; 0 = no contact (incl. same-bin pairs)
CORES = [("sarcomere", "Sarcomere organization (GO:0045214)"), ("cardiomyocyte", "Cardiomyocytes (PanglaoDB)")]

In [9]:
os.makedirs(f"{DATA}/kcores", exist_ok=True)
summary = []
for name, gene_set in CORES:
    genes = [g for g in panel[gene_set] if g in gene_bins]
    module, k = kcore.state_kcore(edge_type, gene_bins, genes, "day80", bin_chrom)
    mat = kcore.state_matrix(edge_type, gene_bins, module, STATE_CODE)
    chroms = [bin_chrom[min(gene_bins[g])] for g in module]
    pd.DataFrame({"gene": module, "chrom": chroms}).to_csv(f"{DATA}/kcores/{name}_genes.csv", index=False)
    pd.DataFrame(mat, index=module, columns=module).to_csv(f"{DATA}/kcores/{name}_states.csv")
    counts = {s: int((mat == c).sum() // 2) for s, c in STATE_CODE.items()}
    summary.append({"core": name, "gene_set": gene_set, "k": k, "n_genes": len(module),
                    "n_chromosomes": len(set(chroms)), **{f"pairs_{s}": n for s, n in counts.items()}})
    print(f"{gene_set}: {k}-core, {len(module)} genes | CM-only {counts['day80']}, PSC-only {counts['day00']}, "
          f"Shared {counts['Shared']}")
pd.DataFrame(summary).to_csv(f"{DATA}/kcores/kcore_summary.csv", index=False)

Sarcomere organization (GO:0045214): 5-core, 10 genes | CM-only 29, PSC-only 0, Shared 7
Cardiomyocytes (PanglaoDB): 17-core, 43 genes | CM-only 474, PSC-only 8, Shared 200


## Replicate agreement (Supplementary Figure 5Sa, panel b)
Spearman correlation, over the 65 pathways of 5c, between the pooled CM − PSC fold difference and that of each replicate
pair (reported in the figure-legend text, not drawn). Writes `data/replicate_agreement.csv`.

In [10]:
# the 65 pathways of 5c in every comparison (pooled + replicate pairs)
pathway_coloc = coloc.query("category == 'hallmark' or gene_set in @panel_names").reset_index(drop=True)
pathway_coloc["diff"] = pathway_coloc.fold_b - pathway_coloc.fold_a
cross_pairs = [c for c, (_, _, kind, _) in COMPARISONS.items() if kind == "cross"]
within_pairs = [c for c, (_, _, kind, _) in COMPARISONS.items() if kind == "within"]
diff_by_comparison = pathway_coloc.pivot(index="gene_set", columns="comparison", values="diff")

agreement = []
for comparison in cross_pairs + within_pairs:
    m = diff_by_comparison[["pooled", comparison]].dropna()
    rho = spearmanr(m.pooled, m[comparison]).statistic
    agreement.append({"comparison": comparison, "spearman_rho_vs_pooled": rho, "n_sets": len(m)})
    print(f"{comparison}: Spearman rho vs pooled = {rho:.2f} (n={len(m)})")
pd.DataFrame(agreement).to_csv(f"{DATA}/replicate_agreement.csv", index=False)

day00_R1 vs day80_R1: Spearman rho vs pooled = 0.94 (n=65)
day00_R1 vs day80_R2: Spearman rho vs pooled = 0.95 (n=65)
day00_R2 vs day80_R1: Spearman rho vs pooled = 0.94 (n=65)
day00_R2 vs day80_R2: Spearman rho vs pooled = 0.95 (n=65)
day00_R1 vs day00_R2: Spearman rho vs pooled = 0.01 (n=65)
day80_R1 vs day80_R2: Spearman rho vs pooled = 0.14 (n=65)
